# 05 · Totals for a country or region, and the traps in them

**What this teaches:** how to get forecast totals for a **country** or an **admin-1 region**, and why to take them from the country and admin-1 files instead of adding up grid cells yourself.

**The short version.** Expected deaths (`sb_mean`) add up: the cells of a country sum to the country (so do the recorded deaths, `sb_actual`, which notebook 01 checks). **No other forecast summary does.** A country's chance of deaths is not the average of its cells' chances, and not their sum; its severe case is not the sum of theirs. The API adds up the draws first and summarises after, which only it can do, because the draws stay on the server. So for a total, ask for the level you need.

Each point below is measured on the data served, not asserted.

**You need an API key** (set in `.env`, see `notebooks/README.md`). The grid file is about 130 MB; only one month of it is read.

## 1 · Setup

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

from views_crafdapi.client import CrafdApiClient
from views_crafdapi.data_checks import DataCheckError, check_levels_add_up
from views_crafdapi.plotting import plot_pixel_grid

load_dotenv()  # reads CRAFD_KEY (your key) and, optionally, VIEWS_API_URL

BASE_URL = os.environ.get("VIEWS_API_URL", "https://crafdapi.viewsforecasting.org")
API_KEY = os.environ.get("CRAFD_KEY", "").strip().strip('"')

client = CrafdApiClient(BASE_URL, API_KEY, timeout=300)
print("Base URL:", BASE_URL)
print("API key :", "set" if API_KEY else "MISSING: set it in .env")

In [ ]:
if not API_KEY or API_KEY == "your-api-key-here":
    raise SystemExit("No usable API key. Set CRAFD_KEY in .env (see notebooks/README.md), restart "
                     "the kernel and run again.")
try:
    health = client.health()
except Exception as e:   # an error answer, a timeout or a dropped connection
    health = {}
    print("Health check unavailable, continuing:", e)
RUN = health.get("served_run") or "not reported"
print("Run served:", RUN)

## 2 · One month at all three levels

The first forecast month, from the three bulk files: grid cells, admin-1 regions and countries. Each grid row says which region and country it is in.

In [ ]:
COLUMNS = ["sb_mean", "sb_p_gt0", "sb_p_gt25", "sb_severe_scenario"]
country_all, MONTH = client.download_bulk("country", "sb")
country = country_all[country_all["month"] == MONTH]
admin1, admin1_month = client.download_bulk("admin1", "sb", [MONTH], ["month", "admin1_code", "admin1_name", "country_code", *COLUMNS])
grid, grid_month = client.download_bulk("grid", "sb", [MONTH],
                                 ["month", "priogrid_id", "latitude", "longitude", "admin1_code", "country_code", *COLUMNS])
if not MONTH == admin1_month == grid_month:   # a new forecast arrived between the downloads
    raise SystemExit(f"The three files come from different forecasts ({MONTH}, {admin1_month}, {grid_month}). Run this cell again.")
nga_rows = country.loc[country["country_iso3"] == "NGA", "country_code"]
NIGERIA = int(nga_rows.iloc[0]) if len(nga_rows) else None   # the example country; the checks do not need it
print(f"Month {MONTH}: {len(grid):,} grid cells, {len(admin1):,} admin-1 regions, {len(country):,} countries")

## 3 · Expected deaths add up

`sb_mean` is the expected number of deaths. The expected value of a sum is the sum of the expected values, so the cells of a region add up to the region, and the cells of a country to the country (up to rounding). This is the one summary you may add up yourself, across places and across months.

The cell below compares every country and every region with its own cells, not only the world's total, so two errors that cancel out still show. (A cell can lie in a country but in no admin-1 region, so regions are compared with their cells, not with the country.)

In [ ]:
try:
    print(check_levels_add_up({"grid": grid, "admin1": admin1, "country": country}, "sb_mean"))
except DataCheckError as problem:
    raise SystemExit(f"The check of expected deaths failed: {problem}")


def place_by_place(cells, served, key):
    """Each place's served expected deaths beside the sum of its own cells'."""
    table = pd.DataFrame({"served": served.set_index(key)["sb_mean"],
                          "cells": cells.groupby(key)["sb_mean"].sum(min_count=1)})
    every_place = len(table)
    table = table.dropna()   # a place with no value served, or no cell with a value, cannot be compared
    table.attrs["left_out"] = every_place - len(table)
    scale = table[["served", "cells"]].abs().max(axis=1).clip(lower=1)
    table["off"] = (table["served"] - table["cells"]).abs() > 1e-5 * scale   # float32 rounding
    return table


by_place = {("country", "countries"): place_by_place(grid, country, "country_code"),
            ("region", "regions"): place_by_place(grid, admin1, "admin1_code")}
for (one, many), table in by_place.items():
    print(f"{many}: {len(table):,} compared with their own cells; not adding up: {int(table['off'].sum())}; "
          f"left out (no value served, or no cell with a value): {table.attrs['left_out']}")
    if table["off"].any():
        first = table[table["off"]].iloc[0]
        raise SystemExit(f"Expected deaths do not add up for {one} {table[table['off']].index[0]}: "
                         f"{first['served']:,.3f} served, {first['cells']:,.3f} from its cells")
if NIGERIA in by_place["country", "countries"].index:
    nga = by_place["country", "countries"].loc[NIGERIA]
    n_cells = int(grid.loc[grid["country_code"] == NIGERIA, "sb_mean"].notna().sum())
    print(f"Nigeria: {nga['served']:,.1f} expected deaths; its {n_cells:,} cells add up to {nga['cells']:,.1f}")

## 4 · The chance of any death does not average

`sb_p_gt0` is the chance of at least one death. For a country it is the chance that the country's total is above zero, so it is **at least as high as its highest cell's**: one cell with deaths is enough. The average of the cells' chances answers a different question, how likely a typical cell is to see deaths, and **understates** the country's chance, often by far.

The ordering holds when no draw is negative. The cell below counts the countries and regions where it does not hold, rather than assuming there are none.

In [ ]:
def compare_chances(cells, served, key, column):
    """Per place at one level: the served chance, its cells' highest chance and their average."""
    by = cells.groupby(key)[column]
    table = pd.DataFrame({"served": served.set_index(key)[column], "highest_cell": by.max(),
                          "average_of_cells": by.mean(), "cells": by.count()})   # cells with a chance
    every_place = len(table)
    table = table.dropna()   # a place with no value served, or no cell with a value, cannot be compared
    table.attrs["left_out"] = every_place - len(table)
    table["breaks_the_ordering"] = table["served"] < table["highest_cell"] - 1e-9
    return table


# A cell with no usable draws has no chances (empty, not 0); those cells are left out of the
# highest and the average, and counted here.
print(f"Grid cells with no chance of any death given (no usable draws): {int(grid['sb_p_gt0'].isna().sum())}")


chances = {"country": compare_chances(grid, country, "country_code", "sb_p_gt0"),
           "admin1": compare_chances(grid, admin1, "admin1_code", "sb_p_gt0")}
for level, table in chances.items():
    gap = table["served"] - table["average_of_cells"]
    print(f"{level:8}: {len(table):,} places; the average of cells understates the served chance by up to "
          f"{gap.max():.2f} (median {gap.median():.2f}); places below their highest cell: "
          f"{int(table['breaks_the_ordering'].sum())}; left out (no value served, or no cell with a value): "
          f"{table.attrs['left_out']}")
row = chances["country"].loc[NIGERIA] if NIGERIA in chances["country"].index else None
if row is not None:
    print(f"Nigeria: chance of any death {row['served']:.2f}; highest cell {row['highest_cell']:.2f}; "
          f"average of its {int(row['cells'])} cells {row['average_of_cells']:.3f}")
chances["country"].query("breaks_the_ordering").head()

## 5 · Chances do not add

Adding the cells' chances does not give the country's chance either, for two reasons. A draw in which two cells both pass the threshold is counted twice, so the sum can pass 1, which no chance can. And a country can pass the threshold when no single cell does (three cells with 10 deaths each pass 25 together), which the cells' chances do not count at all. Here, the chance of more than 25 deaths.

In [ ]:
print(f"Grid cells with no chance of more than 25 deaths given (left out of the sums): {int(grid['sb_p_gt25'].isna().sum())}")
summed = grid.groupby("country_code")["sb_p_gt25"].sum()
over_one = summed[summed > 1]
print(f"Countries whose cells' chances of more than 25 deaths add up to more than 1: {len(over_one)}")
served_by_country = country.set_index("country_code")
if len(over_one) and over_one.idxmax() in served_by_country.index:
    worst = over_one.idxmax()
    print(f"For example {served_by_country.loc[worst, 'country_name']}: the cells add up to {over_one.max():.2f}; "
          f"the country's chance is {served_by_country.loc[worst, 'sb_p_gt25']:.2f}")

## 6 · The severe case does not add

`sb_severe_scenario` is the average of the worst 5% of the draws: what a bad month could realistically look like. Each cell's worst draws are its own; adding the cells' severe cases assumes every cell has its bad month in the same draws. The country's severe case is taken from the country's own worst draws, so it is never above the sum of its cells', and equals it only when the cells' worst draws coincide. In practice the sum **overstates** it. The cell below also counts any country where the sum comes out lower, which should not happen.

In [ ]:
# A cell with fewer than 20 usable draws has no severe case (empty). A country with such a cell
# is left out: summing only its other cells would understate the sum.
by_country = grid.groupby("country_code")["sb_severe_scenario"]
severe = pd.DataFrame({"served": country.set_index("country_code")["sb_severe_scenario"],
                       "sum_of_cells": by_country.sum(min_count=1),
                       "empty_cells": by_country.apply(lambda s: int(s.isna().sum()))})
no_value_served = int(severe["served"].isna().sum())
above_zero = severe[severe["served"] > 0]
severe = above_zero.dropna()
severe = severe[severe["empty_cells"] == 0]
print(f"Countries with no severe case served: {no_value_served}; with one above zero: {len(above_zero)}; "
      f"left out of those (a cell has none, or no cells): {len(above_zero) - len(severe)}")
ratio = severe["sum_of_cells"] / severe["served"]
print(f"Countries compared: {len(severe)}; the sum of the cells' severe cases is "
      f"{ratio.median():.1f} times the country's (median), from {ratio.min():.1f} to {ratio.max():.1f} times")
print(f"Countries where the sum is below the country's severe case: {int((ratio < 1 - 1e-5).sum())}")   # float32 rounding
if NIGERIA in severe.index:
    print(f"Nigeria: severe case {severe.loc[NIGERIA, 'served']:,.0f} deaths; its cells' severe cases add up to "
          f"{severe.loc[NIGERIA, 'sum_of_cells']:,.0f}")

## 7 · Which column to map at which level

At country level the chance of *any* death is close to 1 for many countries (the cell below counts them), so a map of it shows little. The chance of more than 25 deaths, or the expected deaths, separate countries better. At admin-1 level the chance of more than 25 is a good default.

Each map below draws a level's value on its grid cells: every cell takes the value of the region or country it lies in. A chance is drawn without the log scale; deaths are drawn with it.

In [ ]:
near_one = int((country["sb_p_gt0"] >= 0.95).sum())
print(f"Countries with a chance of any death of 0.95 or more: {near_one} of {int(country['sb_p_gt0'].notna().sum())}")

on_grid = grid[["latitude", "longitude", "admin1_code", "country_code"]].copy()
on_grid["region_p_gt25"] = on_grid["admin1_code"].map(admin1.set_index("admin1_code")["sb_p_gt25"])
on_grid["country_mean"] = on_grid["country_code"].map(country.set_index("country_code")["sb_mean"])

fig, ax, _ = plot_pixel_grid(on_grid.dropna(subset=["region_p_gt25"]), lon_col="longitude", lat_col="latitude",
                             value_col="region_p_gt25", transform="none", cmap="YlOrRd", figsize=(14, 7))
ax.set_title(f"Chance of more than 25 state-based deaths, by admin-1 region, {MONTH}  (run {RUN})")
plt.show()

fig, ax, _ = plot_pixel_grid(on_grid.dropna(subset=["country_mean"]), lon_col="longitude", lat_col="latitude",
                             value_col="country_mean", transform="log1p", cmap="YlOrRd", figsize=(14, 7))
ax.set_title(f"Expected state-based deaths, by country, {MONTH} (log scale)  (run {RUN})")
plt.show()

## Next steps

- **`02_visualization.ipynb`**: the same data as grid-cell maps.
- The [data dictionary](../docs/api/data_dictionary.md), section "Adding up places or months yourself", says the same as this notebook in words.